# MOC set algebra on morton covers

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/espg/mortie/HEAD?labpath=examples%2Fmorton_set_algebra.ipynb)

`mortie` represents a region on the sphere as a **morton cover** — a set of
HEALPix cells, each packed into a single unsigned 64-bit *morton index* that
self-encodes its order (resolution) and ancestry. A **Multi-Order Coverage**
(MOC) is just such a cover at mixed orders: coarse cells in the interior, fine
cells along the boundary.

Because a cover is a *set*, the natural way to combine two regions is **set
algebra**. `mortie` exposes five boolean verbs, all backed by the `healpix`
crate's BMOC and normalized to mortie's canonical compact form:

| verb | meaning | result |
|------|---------|--------|
| `moc_or(a, b)`    | union | cells in **a or b** |
| `moc_and(a, b)`   | intersection | cells in **a and b** |
| `moc_minus(a, b)` | difference | cells in **a but not b** |
| `moc_xor(a, b)`   | symmetric difference | cells in **exactly one** of a, b |
| `moc_not(a, domain)` | domain-bounded complement | cells of **domain not in a** |

This notebook explains each verb, shows it on small covers, and visualizes the
**before / after** so you can see exactly which cells survive. It is
[runnable on Binder](https://mybinder.org/v2/gh/espg/mortie/HEAD?labpath=examples%2Fmorton_set_algebra.ipynb)
(see `binder/` for the repo2docker config).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection
import mortie

mortie.__version__

'0.8.1'

## A plotting helper

A morton cell decodes to its four corner vertices on the sphere via
`mortie.mort2polygon` (returned as `[lat, lon]` rows), and to its HEALPix
`(nested-pixel, order)` via `mortie.mort2healpix`. The helper below turns a
cover into matplotlib polygon patches in plain longitude/latitude axes — enough
to read off a set operation at a glance. (For publication-quality spherical
plots the other notebooks in `examples/` use `cartopy`; we keep this one
dependency-light so the Binder image stays small.)

In [2]:
def cover_patches(cover):
    """(lon, lat) matplotlib patches + per-cell order for a morton cover."""
    patches, orders = [], []
    for m in np.atleast_1d(cover):
        latlon = np.asarray(mortie.mort2polygon(int(m)))   # [lat, lon] rows
        patches.append(MplPolygon(np.column_stack([latlon[:, 1], latlon[:, 0]]),
                                  closed=True))
        _, order = mortie.mort2healpix(int(m))
        orders.append(order)
    return patches, np.asarray(orders)


def draw_cover(ax, cover, facecolor, alpha=0.55, label=None):
    patches, _ = cover_patches(cover)
    ax.add_collection(PatchCollection(patches, facecolor=facecolor,
                                      edgecolor='0.3', linewidth=0.4, alpha=alpha))
    if label is not None:
        ax.plot([], [], 's', color=facecolor, alpha=alpha, label=label)


EXTENT = (8, 27, 38, 54)   # lon_min, lon_max, lat_min, lat_max


def setup_ax(ax, title):
    ax.set_xlim(EXTENT[0], EXTENT[1])
    ax.set_ylim(EXTENT[2], EXTENT[3])
    ax.set_aspect('equal')
    ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
    ax.set_title(title)


def show_op(title, a, b, result, result_color='#2ca02c'):
    """Before (a, b) | after (result), with a, b outlined for context."""
    fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.6))
    draw_cover(left, a, '#1f77b4', label='a')
    draw_cover(left, b, '#d62728', label='b')
    setup_ax(left, 'inputs: a (blue), b (red)'); left.legend(loc='upper left')
    draw_cover(right, a, '#1f77b4', alpha=0.12)
    draw_cover(right, b, '#d62728', alpha=0.12)
    draw_cover(right, result, result_color, label='result')
    setup_ax(right, title); right.legend(loc='upper left')
    fig.tight_layout(); plt.show()

## Two overlapping covers

We cover two overlapping lat/lon squares with `morton_coverage` at order 5. Each
call returns a flat `uint64` array of morton indices whose cells tile the
polygon. `a` and `b` overlap in the middle — that overlap is what the set
operations key on.

In [3]:
a = mortie.morton_coverage(np.array([40., 40., 48., 48., 40.]),
                           np.array([10., 20., 20., 10., 10.]), order=5)
b = mortie.morton_coverage(np.array([44., 44., 52., 52., 44.]),
                           np.array([15., 25., 25., 15., 15.]), order=5)
len(a), len(b), a.dtype

(29, 29, dtype('uint64'))

In [4]:
fig, ax = plt.subplots(figsize=(6, 4.6))
draw_cover(ax, a, '#1f77b4', label='a')
draw_cover(ax, b, '#d62728', label='b')
setup_ax(ax, 'two overlapping covers'); ax.legend(loc='upper left')
plt.show()

## `moc_or` — union

`moc_or(a, b)` returns every cell covered by **either** operand. Where the two
covers meet, complete sibling quartets collapse into their coarser parent, so
the union is returned in mortie's canonical compact (mixed-order) form rather
than as a redundant flat list.

In [5]:
u = mortie.moc_or(a, b)
print('union has', len(u), 'cells')
show_op('moc_or(a, b)  =  a ∪ b', a, b, u)

union has 20 cells


## `moc_and` — intersection

`moc_and(a, b)` keeps only cells covered by **both** operands: the overlap
region in the middle.

In [6]:
i = mortie.moc_and(a, b)
print('intersection has', len(i), 'cells')
show_op('moc_and(a, b)  =  a ∩ b', a, b, i)

intersection has 8 cells


## `moc_minus` — difference

`moc_minus(a, b)` keeps cells in **a but not b** — it carves b's footprint out
of a. Unlike the others it is *not* symmetric: `minus(a, b) != minus(b, a)`.

In [7]:
d = mortie.moc_minus(a, b)
print('a minus b has', len(d), 'cells')
show_op('moc_minus(a, b)  =  a \\ b', a, b, d)

a minus b has 15 cells


## `moc_xor` — symmetric difference

`moc_xor(a, b)` keeps cells in **exactly one** operand — the union minus the
intersection, `(a ∪ b) \ (a ∩ b)`. It is the cells that *changed*: think of
`a` and `b` as the same region at two different times, and `xor` is everything
that was gained **or** lost. The shared middle drops out.

In [8]:
x = mortie.moc_xor(a, b)
print('a xor b has', len(x), 'cells')
# identity check: xor == or minus and
assert np.array_equal(np.sort(x),
                      np.sort(mortie.moc_minus(mortie.moc_or(a, b),
                                               mortie.moc_and(a, b))))
show_op('moc_xor(a, b)  =  (a ∪ b) \\ (a ∩ b)', a, b, x)

a xor b has 27 cells


## `moc_not` — domain-bounded complement

Complement only makes sense against a **domain** to complement *within*.
`moc_not(cover, domain)` returns the cells of `domain` not in `cover` — it is
exactly `moc_minus(domain, cover)`. The typical use is a **shard**: a coarse
cell with finer cells enumerated inside it, where `not` returns the cells *not
yet* enumerated within that shard.

Here we use the union `a ∪ b` as the domain and complement `a` within it, so
the result is "everything in the combined region that is **not** in `a`" — which
is just `b \ a`.

In [9]:
domain = mortie.moc_or(a, b)
n = mortie.moc_not(a, domain=domain)
print('not(a, domain) has', len(n), 'cells')
assert np.array_equal(np.sort(n), np.sort(mortie.moc_minus(domain, a)))
show_op('moc_not(a, domain=a∪b)  =  (a∪b) \\ a', a, b, n, result_color='#9467bd')

not(a, domain) has 12 cells


With no `domain` argument, `moc_not(cover)` complements against the **whole
sphere** (the 12 order-0 HEALPix base cells). Out-of-domain cells in `cover` are
clipped, and a `UserWarning` is raised so the clip is never silent.

In [10]:
whole = mortie.moc_not(a)        # complement of a over the entire sphere
print('whole-sphere complement of a has', len(whole), 'cells')

whole-sphere complement of a has 43 cells


## Notes

- **Orders.** The point/encode path (`geo2mort`, `norm2mort`) reaches order 29,
  but the MOC set-algebra and coverage paths currently operate up to **order
  18** (the BMOC depth ceiling). Raising that ceiling is tracked in
  [#61](https://github.com/espg/mortie/issues/61) and
  [#60](https://github.com/espg/mortie/issues/60).
- **Covers are plain `uint64` arrays.** Each morton index self-encodes its
  order, so a mixed-order MOC needs no side-channel — `np.sort` on the raw words
  is the Z-order.
- **API:** `moc_or`, `moc_and`, `moc_minus`, `moc_xor`, `moc_not`,
  `morton_coverage`, `morton_coverage_moc`, `moc_to_order`.